# Instrucciones para generar .json para evaluación

Estructura general: 
1. Guardar resultados de un modelo en un DataFrame de Pandas. 
2. Exportar ese DataFrame a un archivo .json.
3. Pasar ese archivo .json junto al archivo de evaluación a la paquetería de PyEvALL.

## Paso 1 y 2: Creación del DataFrame (RESULTADOS DE LOS MODELOS) y del archivo .json

En la vida real este DataFrame se obtiene de los resultados **REALES** del sistema que estemos trabajando. En este notebook creamos un dataframe sintético de manera aleatoria para que sirva de ejemplo de la estructura.

El dataframe debe de tener 3 columnas:
1. `test_case` Identificador para este task. SIEMPRE DEBE DE SER EL VALOR "EXIST2025"
2. `id` Valor numérico extraído del conjunto de datos inicial. Es importante que sea consistente ya que debe de emparejarse con el conjunto de datos que tiene los valores de verdad.
3. `value` La predicción del modelo. Hay dos variantes del posible valor aquí, en función del task. Favor de ver los detalles en el código.

El dataframe se debe de guardar usando la función `.to_json()` de pandas. IMPORTANTE: USEN EL PARÁMETRO `orient = 'records'`.

In [ ]:
# Creación de conjuntos de datos sintéticos.
import pandas as pd
import random

# Columna test_case
test_case = ["EXIST2025" for i in range(10)]

# Columna id
id = [f"30000{i}" for i in range(10)]

# Columna value: SUBTASK 2.1 Y 2.2
value = ["NO" if random.random() > 0.5 else "YES" for i in range(10)]

# Columna value: SUBTASK 2.3
rand_labels = ["NO", "IDEOLOGICAL-INEQUALITY", "STEREOTYPING-DOMINANCE", "OBJECTIFICATION", "SEXUAL-VIOLENCE", "MYSOGYNY-NON-SEXUAL-VIOLENCE"]
value_sub23 = [random.sample(rand_labels) for i in range(10)]


def create_and_json(test_list, id_list, value_list, sub_23):
    test_dictionary = {"test_case": test_list, "id": id_list, "value": value_list}
    df = pd.DataFrame(dataframe = test_dictionary)

    # Guardamos dos veces el mismo conjunto de datos ya que uno va a servir como el Gold Standard (dígase el conjunto de datos de prueba.)
    # USEN PARÁMETRO DE orient = 'records'
    # Pandas permite guardar los json bajo distintos formatos de orientación, pero esos formatos NO FUNCIONAN PARA PYEVALL.
    if not sub_23:
        df.to_json(f'/home/flopezp/GIL-Exist-2026/Archivo de salida/test_outputs/Subtask\ 2.1\ y\ 2.2/pred/pred.json', orient = 'records')
        df.to_json(f'/home/flopezp/GIL-Exist-2026/Archivo de salida/test_outputs/Subtask\ 2.1\ y\ 2.2/gold/gold.json', orient = 'records')
    else:
        df.to_json(f'/home/flopezp/GIL-Exist-2026/Archivo de salida/test_outputs/Subtask\ 2.3/pred/pred.json', orient = 'records')
        df.to_json(f'/home/flopezp/GIL-Exist-2026/Archivo de salida/test_outputs/Subtask\ 2.3/gold/gold.json', orient = 'records')

## Paso 3: Uso de PyEvALL

En el pdf del EXIST 2026 viene una implementación de la evaluación (La misma que está aquí salvo unos ajustes)

In [1]:
from pyevall.evaluation import PyEvALLEvaluation
from pyevall.utils.utils import PyEvALLUtils

def predicciones(pred_dir, gold_dir, subtask):
    assert subtask in [1,2,3], "Elegir subtask válido"
    if subtask == 1:
        print("Evaluando: Subtask 1")
    if subtask == 2:
        print("Evaluando: Subtask 2")
    else:
        print("Evaluando: Subtask 3")

    predictions = pred_dir
    gold = gold_dir
    test = PyEvALLEvaluation()
    params= dict()
    params[PyEvALLUtils.PARAM_REPORT]= PyEvALLUtils.PARAM_OPTION_REPORT_EMBEDDED
    metrics=["ICM", "ICMNorm" ,"FMeasure"]
    if subtask == 2:
        TASK2_2_HIERARCHY = {"YES":["DIRECT", "JUDGEMENTAL"], "NO":[]}
        params[PyEvALLUtils.PARAM_HIERARCHY]= TASK2_2_HIERARCHY
    if subtask == 3:
        TASK2_3_HIERARCHY = {
            "YES":["IDEOLOGICAL-INEQUALITY","STEREOTYPING-DOMINANCE","OBJECTIFICATION", "SEXUAL-VIOLENCE", "MISOGYNY-NON-SEXUAL-VIOLENCE"], 
            "NO":[]
            }
        params[PyEvALLUtils.PARAM_HIERARCHY]= TASK2_3_HIERARCHY
    report= test.evaluate(predictions, gold, metrics, **params)
    report.print_report()
    print("===================================")

In [3]:
subtask2_1 = '/home/flopezp/GIL-Exist-2026/Archivo de salida/dev_runs/exist_dataset/EXIST2025_training_task2_1_gold_hard.json'
subtask2_3 = '/home/flopezp/GIL-Exist-2026/Archivo de salida/dev_runs/exist_dataset/EXIST2025_training_task2_3_gold_hard.json'

predicciones('/home/flopezp/GIL-Exist-2026/Archivo de salida/dev_runs/miranda_dataset/T3_DS_experts_fixed.json', subtask2_3, 3)
predicciones('/home/flopezp/GIL-Exist-2026/Archivo de salida/dev_runs/miranda_dataset/T3_DS_library_fixed.json', subtask2_3, 3)
predicciones('/home/flopezp/GIL-Exist-2026/Archivo de salida/dev_runs/miranda_dataset/T3_DS_med_fixed.json', subtask2_3, 3)

Evaluando: Subtask 3
2026-04-24 16:55:35,180 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-04-24 16:55:35,322 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-04-24 16:55:35,702 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Normalized evaluation method
2026-04-24 16:55:35,703 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-04-24 16:55:36,083 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-04-24 16:55:36,762 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
{
  "metrics": {
    "ICM": {
      "name": "Information Contrast model",
      "acronym": "ICM",
      "description": "Coming soon!",
      "status": "OK",
      "results": {
        "test_cases": [{
          "name": "EXIST2025",
        